# Penjelasan Per Cell - Berdasarkan Isi PPT "Telling AI What It Doesn't Know: From NLP to RAG"

## CELL 1 - Setup  (CORE)

Just run it. Nothing is installed.

In [1]:
# Nothing is installed. Colab already has everything we need.
import sentence_transformers

TIMINGS = {}


def _probe_answer_engine():
    """Work out which model will write our answers, and say why.

    Reported at the start rather than at the end, so the room knows which path it
    is on from minute one instead of discovering it two hours in.
    """
    try:
        from google.colab import ai
    except Exception as exc:
        return "offline model (slower)", f"cannot import google.colab.ai ({type(exc).__name__})"
    if not hasattr(ai, "generate_text"):
        names = ", ".join(n for n in dir(ai) if not n.startswith("_"))
        return "offline model (slower)", f"google.colab.ai has no generate_text; it offers: {names}"
    return "Colab AI (fast)", ""


print("SETUP OK - sentence-transformers", sentence_transformers.__version__)

_engine_name, _engine_reason = _probe_answer_engine()
print("Answer engine:", _engine_name)
if _engine_reason:
    print("  reason:", _engine_reason)
    print("  This still works. Answers just take a little longer.")

print("\nShow a GREEN paper if you can see SETUP OK.")

SETUP OK - sentence-transformers 5.7.0
Answer engine: Colab AI (fast)

Show a GREEN paper if you can see SETUP OK.


menjelaskan bahwa AI/neural network melalui dua tahap utama: Training (belajar dari dataset) dan Inference (memproses prompt untuk menghasilkan jawaban real-time). Cell ini adalah persiapan sebelum masuk ke tahap inference tersebut - menyiapkan alat yang nanti dipakai untuk "berbicara" dengan AI.

## CELL 2 - Load the handbook  (CORE)

35 paragraphs from a fictional college. Run it, do not open it.

In [19]:
# @title Run this cell to load the handbook (35 paragraphs) { display-mode: "form" }
# Riverstone Polytechnic is a fictional college. Nothing here is real.
PARAGRAPHS = [
    "Roblox is an online platform launched in 2006 that lets users create, share, and play games built by other users, called experiences. Roblox is available on Windows, Mac, iOS, Android, and Xbox, with cross-platform play between most of these devices. The platform hosts millions of user-created experiences, and Roblox itself does not develop most of the games. Users interact through a customizable 3D avatar.",
    "Robux is the virtual currency of Roblox, used to buy avatar items, game passes, and access to private servers. Robux cannot be transferred directly to another player's account for free; buying or receiving Robux is regulated by Roblox's Terms of Service. Robux purchased with real money can be spent inside experiences or in the avatar Catalog.",
    "Robux can be purchased in fixed packages, for example around 400 Robux for 4.99 USD, 800 Robux for 9.99 USD, 1,700 Robux for 19.99 USD, 4,500 Robux for 49.99 USD, and 10,000 Robux for 99.99 USD. Prices and package sizes can change over time and vary by region and platform.",
    "Roblox Premium has three common tiers. Premium 450 gives 450 Robux per month for about 4.99 USD. Premium 1000 gives 1,000 Robux per month for about 9.99 USD. Premium 2200 gives 2,200 Robux per month for about 19.99 USD. Premium members also receive a 10 percent bonus on every Robux purchase.",
    "Premium membership unlocks more than monthly Robux. A Premium member can trade Limited items with other Premium members and resell certain items on the marketplace. Players without Premium cannot trade or resell items at all.",
    "Developer Exchange, or DevEx, lets qualifying creators convert Robux earned in their experiences into real currency. To qualify, a creator must be at least 13 years old, have Roblox Premium, and hold a minimum balance, historically around 30,000 Robux, before requesting an exchange. The DevEx rate has historically been close to 0.0035 USD per Robux.",
    "Roblox Studio is the free desktop tool used to build experiences on Roblox. It uses a scripting language called Luau, a fast variant of Lua, and lets creators build 3D worlds, write game logic, and design user interfaces without needing a separate game engine.",
    "In Roblox Studio, code can run in a Script, which executes on the server, or a LocalScript, which executes on each player's device. Server scripts control shared game state, while local scripts handle things only relevant to one player, like camera movement or a private UI.",
    "Roblox requires every account to have a birthdate, and features are restricted based on age. Accounts believed to belong to users under 13 have a more limited chat system by default, restricted to a set of pre-approved messages instead of free text.",
    "Parents can set a Parental PIN on a Roblox account to restrict privacy settings, spending limits, and the kinds of experiences the account can access. The PIN must be entered again to change any protected settings.",
    "Groups on Roblox let multiple users organize around a shared brand or experience. A Group can hold its own Robux balance, called Group funds, and the owner can configure what percentage of an experience's earnings goes into the Group fund versus individual members.",
    "Limited items are avatar items with a fixed, capped supply, marked as Limited or Limited Unique. Their resale prices are driven by supply and demand between players, and Roblox displays a Recent Average Price, or RAP, calculated from recent completed trades.",
    "Trading Limited items requires both accounts involved to have Roblox Premium. Roblox's trade system checks that a trade is reasonably balanced in value on both sides before allowing it to complete, to reduce scams.",
    "A Game Pass is a one-time purchase that unlocks a permanent perk inside a specific experience, such as a VIP area or extra inventory space. Game Pass prices are set by the creator, with a typical minimum price around 5 to 20 Robux depending on Roblox's current rules.",
    "A Developer Product is different from a Game Pass because it can be purchased more than once by the same player, making it suited for consumable items like in-game currency, potions, or extra lives.",
    "When a player spends Robux inside an experience, Roblox takes a marketplace cut before the rest goes to the creator. Historically, creators have kept roughly 30 percent of the Robux spent through the standard marketplace flow, though the exact split can depend on the type of transaction.",
    "Private servers, also called VIP servers, let a group of players access an instance of an experience that only invited people can join. The experience's creator sets a monthly Robux price for renting a private server.",
    "Roblox Corporation became a public company through a direct listing on the New York Stock Exchange in March 2021, trading under the ticker symbol RBLX.",
    "Roblox reports tens of millions of daily active users across the platform worldwide, with usage spread across many age groups, though the platform is especially popular among younger players.",
    "Some of the most-played experiences on Roblox in recent years include Adopt Me!, Brookhaven RP, Blox Fruits, Pet Simulator, and Blade Ball, each created and maintained by independent development teams rather than Roblox itself.",
    "A Roblox avatar can be customized through the Catalog, which sells hats, faces, gear, animations, and full outfit bundles. Avatars can use the classic block-style rig or the more articulated R15 rig, which supports more detailed animations.",
    "Roblox uses an automated text filter across chat and usernames to block personal information, such as phone numbers and addresses, along with inappropriate language. Filtering rules are generally stricter for accounts registered as under 13.",
    "Roblox Studio has a plugin marketplace where creators can find free and paid tools that speed up building, from terrain generators to animation editors. Plugins are separate from Game Passes and Developer Products, which are sold inside published experiences rather than inside Studio.",
    "Every year, Roblox hosts the Bloxy Awards, an event that recognizes top creators, popular experiences, and community achievements from the past year, similar to an awards show for the platform's creator community.",
    "Roblox has a dedicated Trust and Safety team along with an in-experience reporting system, allowing players to report other users or experiences for violating Roblox's Community Standards.",
    "Roblox experiences are subject to Roblox's Experience Guidelines, which restrict content such as graphic violence, real-world gambling, and explicit material. Experiences that violate these guidelines can be removed from the platform.",
    "Creators can earn Robux not only from Game Passes and Developer Products, but also from Immersive Ads and, for qualifying creators, from Engagement-Based Payouts, where Roblox shares revenue based on how much time Premium subscribers spend in their experience.",
    "Roblox is free to download and free to create an account, though spending real money is required to buy Robux, Premium membership, or certain items directly. A player can enjoy and play most experiences on the platform without ever spending money.",
]

menyebut use-case seperti company-specific information, updated laws and regulations, dan private/protected data sebagai contoh pengetahuan yang tidak ada di dalam training model, sehingga perlu disediakan lewat sumber eksternal. Handbook 35 paragraf ini adalah contoh nyata dari "sumber eksternal" tersebut.

## CELL 3 - Load the model  (CORE)

Run it and wait for MODEL OK.

In [3]:
# The model that turns text into numbers. About 91 MB, downloaded once.
import time

from sentence_transformers import SentenceTransformer

_started = time.perf_counter()
model = SentenceTransformer("all-MiniLM-L6-v2")
TIMINGS["load model"] = time.perf_counter() - _started

print(f"MODEL OK - ready in {TIMINGS['load model']:.1f} seconds")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MODEL OK - ready in 7.5 seconds


menjelaskan bahwa embedding "dilakukan oleh model lain yang sudah belajar memetakan teks ke vektor" dan "memetakan teks asli ke ruang matematis". Cell ini memuat model tersebut ke dalam notebook.

## CELL 4 - Text becomes numbers  (CORE)

Each sentence becomes a long list of numbers, called an **embedding**.

**Guess first:** how many numbers does one sentence become?

In [4]:
guess = 100        # <-- change this number to your guess

SENTENCES = [
    "I love cats.",
    "I like kittens.",
    "My dog is very friendly.",
    "The stock market fell sharply today.",
    "Interest rates went up again.",
]

vectors = model.encode(SENTENCES, normalize_embeddings=True)
print("you guessed:", guess)
print("real answer:", vectors.shape[1], "numbers per sentence")

you guessed: 100
real answer: 384 numbers per sentence


menyatakan embedding "mengubah makna menjadi bentuk vektor" dan berdimensi "jauh lebih kecil" dibanding one-hot encoding (yang butuh 50.000 angka). Contoh nyata di PPT: King menjadi [-0.32, -0.88, 0.22, 0.25, 0.23, ...]. Cell ini menunjukkan hal yang sama, hanya dengan model dan jumlah dimensi yang berbeda.

## CELL 5 - Similar meaning means a high score  (CORE)

We compare **I love cats.** against all five sentences.

**Guess first:** which one of the SENTENCES gets the highest score?

In [5]:
guess = "My dog is very friendly."

query = "I love cats."
query_vector = model.encode([query], normalize_embeddings=True)[0]
scores = vectors @ query_vector

print("you guessed:", guess)
print(f"\nCompared with: {query}\n")
for sentence, score in sorted(zip(SENTENCES, scores), key=lambda pair: -pair[1]):
    print(f"  {score:+.2f}   {sentence}")

you guessed: My dog is very friendly.

Compared with: I love cats.

  +1.00   I love cats.
  +0.78   I like kittens.
  +0.39   My dog is very friendly.
  +0.06   Interest rates went up again.
  +0.04   The stock market fell sharply today.


menjelaskan lewat contoh "King - Man + Woman ≈ Queen" bahwa kata-kata dengan hubungan makna serupa punya arah/posisi vektor yang mirip. PPT juga menyebut "king ↔ queen" dan "man ↔ woman" sebagai pasangan dengan pola kemiripan yang sama. Cell ini adalah bukti praktis dari prinsip tersebut: kalimat bermakna dekat mendapat skor kemiripan lebih tinggi.

## CELL 6 - Your own sentence  (BONUS)

Change the sentence, run, and see where it lands.

In [6]:
my_sentence = "Small cats are the best animals."   # <-- change this

my_vector = model.encode([my_sentence], normalize_embeddings=True)[0]
for sentence, score in sorted(zip(SENTENCES, vectors @ my_vector), key=lambda pair: -pair[1]):
    print(f"  {score:+.2f}   {sentence}")

  +0.65   I love cats.
  +0.61   I like kittens.
  +0.40   My dog is very friendly.
  +0.04   The stock market fell sharply today.
  +0.01   Interest rates went up again.


menjelaskan lewat contoh King-Queen-Man-Woman bahwa kata/kalimat dengan makna berdekatan akan memiliki posisi vektor yang berdekatan pula, dan hubungan ini bisa diverifikasi secara matematis (seperti pada contoh "King - Man + Woman ≈ Queen"). Cell ini menerapkan prinsip yang sama secara interaktif - kamu memasukkan kalimat sendiri, lalu sistem menunjukkan seberapa dekat posisi vektornya dibanding lima kalimat pembanding, sebagai cara membuktikan sendiri konsep kemiripan makna yang dijelaskan di slide tersebut.

## CELL 7 - Turn the whole handbook into numbers  (CORE)

35 paragraphs. Takes a few seconds. Wait for the shape to print.

In [25]:
import time

_started = time.perf_counter()
PARAGRAPH_VECTORS = model.encode(PARAGRAPHS, normalize_embeddings=True)
TIMINGS["embed handbook"] = time.perf_counter() - _started

print("shape:", PARAGRAPH_VECTORS.shape, f"in {TIMINGS['embed handbook']:.1f} seconds")
print("35 paragraphs, 384 numbers each. Show a GREEN paper if you see (35, 384).")

shape: (28, 384) in 1.3 seconds
35 paragraphs, 384 numbers each. Show a GREEN paper if you see (35, 384).


menunjukkan tahap sebelum "folder pencarian" di mana kumpulan dokumen (database) sudah disiapkan lebih dulu. Cell ini adalah proses mengubah seluruh handbook menjadi bentuk angka (embedding) sebelum bisa dicari.

## CELL 8 - The search function  (CORE)

Read it and run it. Three lines do the work: turn the question into numbers, give every paragraph a score, keep the best ones.

In [26]:
# Read it, run it, do not change it.
def search(question, k=3):
    """Return the k handbook paragraphs closest in meaning to the question."""
    question_vector = model.encode([question], normalize_embeddings=True)[0]
    scores = PARAGRAPH_VECTORS @ question_vector      # one score for every paragraph
    best = scores.argsort()[::-1][:k]                 # highest score first
    return [(int(i) + 1, float(scores[i]), PARAGRAPHS[i]) for i in best]


def show(hits):
    """Print search results in a readable way."""
    for number, score, text in hits:
        print(f"score {score:+.2f}   paragraph {number}:  {text[:85]}...")


print("search() and show() are ready.")

search() and show() are ready.


menunjukkan kotak bergambar folder + kaca pembesar sebagai tahap pencarian sebelum prompt dikirim ke LLM - inilah yang disebut "Retrieval" dalam Retrieval-Augmented Generation. Cell ini adalah implementasi dari kotak tersebut.

## CELL 9 - Ask your own question  (CORE)

Change the question, run it, then tell your partner **which paragraph number** came back.

The model only understands English today, so ask in English. Ideas:

- How many books can I borrow?
- When is the library open on Saturday?
- Can I eat in the computer lab?
- How long is the internship?
- What happens if I fail a subject three times?
- Who is Bytes?

In [27]:
my_question = "How many books can I borrow?"   # <-- change this

show(search(my_question, k=3))

score +0.21   paragraph 13:  Trading Limited items requires both accounts involved to have Roblox Premium. Roblox'...
score +0.20   paragraph 14:  A Game Pass is a one-time purchase that unlocks a permanent perk inside a specific ex...
score +0.20   paragraph 12:  Limited items are avatar items with a fixed, capped supply, marked as Limited or Limi...


menunjukkan kotak folder + kaca pembesar sebagai tahap "Retrieval" - proses mencari paragraf yang paling relevan dari database sebelum dikirim ke LLM. Cell ini adalah kesempatan mempraktikkan langsung tahap Retrieval tersebut: kamu memasukkan pertanyaan sendiri, lalu fungsi search() menunjukkan paragraf handbook mana yang dianggap paling relevan, sesuai mekanisme yang digambarkan di diagram slide.

## CELL 10 - Search always answers, even when it should not  (CORE)

The handbook says nothing about France.

**Guess first:** will the scores be high or low? Write `"high"` or `"low"`.

In [28]:
guess = "high"     # <-- "high" or "low"

print("you guessed:", guess, "\n")
show(search("What is the capital of France?", k=3))
print("\nLesson: search ALWAYS returns something. Read the score.")

you guessed: high 

score +0.11   paragraph 2:  Robux is the virtual currency of Roblox, used to buy avatar items, game passes, and a...
score +0.09   paragraph 6:  Developer Exchange, or DevEx, lets qualifying creators convert Robux earned in their ...
score +0.09   paragraph 7:  Roblox Studio is the free desktop tool used to build experiences on Roblox. It uses a...

Lesson: search ALWAYS returns something. Read the score.


menjelaskan bahwa hallucination "biasanya terjadi ketika model tidak tahu atau bingung, sehingga ia mengarang". PPT juga menyatakan "model tidak tahu bahwa ia tidak tahu" (the model does not know that it does not know). Cell ini menunjukkan gejala serupa pada tahap pencarian: sistem tetap memberi hasil meski topiknya tidak ada di handbook, hanya dengan skor yang rendah.

## CELL 11 - How many paragraphs?  (BONUS)

Change `k` to 1, then to 8. What changes, and what does not?

In [29]:
show(search(my_question, k=1))   # <-- try 1, then try 8

score +0.21   paragraph 13:  Trading Limited items requires both accounts involved to have Roblox Premium. Roblox'...


menyebut "Hybrid search & reranking" sebagai pengembangan lanjutan untuk "pencarian yang lebih akurat" (for more accurate retrieval). Cell ini menyentuh isu yang sama dari sisi paling dasar: parameter k menentukan berapa banyak paragraf yang diambil dari hasil pencarian. Semakin banyak paragraf yang diambil, semakin besar juga kebutuhan untuk menyaring/mengurutkan ulang mana yang benar-benar relevan - itulah persoalan yang coba dijawab oleh reranking di sistem RAG yang lebih matang, sesuai yang disinggung di slide bonus.

## CELL 12 - Put the paragraphs into a prompt  (CORE)

Run it and **read the text it prints**. That text is everything the model will see. This is the whole idea.

In [30]:
def build_prompt(question, hits):
    """Glue the found paragraphs and the question into one piece of text."""
    context = "\n\n".join(f"[Paragraph {n}] {t}" for n, s, t in hits)
    return (
        "Use ONLY the handbook text below to answer the question.\n"
        "If the answer is not in the text, say: I cannot find this in the handbook.\n"
        "Always say which paragraph number you used.\n\n"
        f"Handbook:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


example = "How many books can I borrow?"
print(build_prompt(example, search(example, k=1)))

Use ONLY the handbook text below to answer the question.
If the answer is not in the text, say: I cannot find this in the handbook.
Always say which paragraph number you used.

Handbook:
[Paragraph 13] Trading Limited items requires both accounts involved to have Roblox Premium. Roblox's trade system checks that a trade is reasonably balanced in value on both sides before allowing it to complete, to reduce scams.

Question: How many books can I borrow?
Answer:


menunjukkan kotak "PROMPT+" yang muncul setelah tahap pencarian database, sebelum dikirim ke LLM - menggambarkan prompt asli yang "ditambah" dengan hasil pencarian. Cell ini adalah implementasi dari kotak PROMPT+ tersebut.

## CELL 13 - The answer function  (CORE)

Run it. Do not change it.

In [31]:
# Read it, run it, do not change it.
# First choice is the Gemini model built into Colab: no key, no download, instant.
# If your account cannot use it, this falls back to a small offline model and says so.
import time

_local = {}
_engine = {"reported": None}


def _local_answer(prompt):
    if not _local:
        print("   loading a small offline model, about 300 MB, one time only...")
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        name = "google/flan-t5-small"
        _local["tokenizer"] = AutoTokenizer.from_pretrained(name)
        _local["model"] = AutoModelForSeq2SeqLM.from_pretrained(name)
    tokenizer, small_model = _local["tokenizer"], _local["model"]
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output = small_model.generate(**inputs, max_new_tokens=48)
    return tokenizer.decode(output[0], skip_special_tokens=True)


def _report(engine, reason=""):
    """Announce the engine once, with the exact reason for any fallback.

    Naming the reason matters: a renamed API and an ineligible account both raise,
    and without this they would look identical.
    """
    if _engine["reported"] != engine:
        _engine["reported"] = engine
        print(f"[engine: {engine}]" + (f" {reason}" if reason else ""))


def answer(prompt):
    """Send a prompt to a language model and return its text."""
    started = time.perf_counter()
    try:
        from google.colab import ai

        text = ai.generate_text(prompt)
        _report("Colab AI")
    except Exception as exc:
        _report("offline model", f"{type(exc).__name__}: {str(exc)[:140]}")
        text = _local_answer(prompt)
    TIMINGS["last answer"] = time.perf_counter() - started
    return text


def ask(question, k=3):
    """The whole system: search, build the prompt, then answer."""
    hits = search(question, k=k)
    return answer(build_prompt(question, hits)), hits


print("answer() and ask() are ready.")

answer() and ask() are ready.


menunjukkan alur lengkap: PROMPT → pencarian database → PROMPT+ → layar komputer (LLM) → ANSWER. Fungsi ask() di cell ini menjalankan seluruh alur tersebut sekaligus.

## CELL 14 - Ask WITHOUT the handbook  (CORE)

Five rules in this handbook were invented for today. No chatbot in the world has ever seen them.

**Guess first:** What the model says when there is no handbook at all? will it say "I do not know", or will it invent a number? Write `"know"` or `"invent"`.

![no_RAG](https://raw.githubusercontent.com/modafarshouha/mini-rag-workshop/main/assets/no_RAG.png)

In [32]:
question = "What is the minimum Robux balance needed to request a Developer Exchange payout?"

guess = "know"     # <-- "know" or "invent"

print("you guessed:", guess, "\n")

# No handbook. We insist on a number, the way many real applications do.
prompt_without_handbook = (
    f"Question about Roblox: {question}\n"
    "Reply with one number (Robux) and nothing else. "
    "Do not ask me for more information."
)

print(prompt_without_handbook)
print()
print(answer(prompt_without_handbook))

you guessed: know 

Question about Roblox: What is the minimum Robux balance needed to request a Developer Exchange payout?
Reply with one number (Robux) and nothing else. Do not ask me for more information.

[engine: Colab AI]
30,000


menjelaskan bahwa "prediksi dibatasi oleh apa yang dipelajari model saat training" dan "model tidak tahu apa yang tidak pernah dilihatnya". PPT juga memakai analogi: "seseorang yang membaca semua buku di perpustakaan tapi tidak ingat buku mana bilang apa - bisa menulis lancar, tapi bisa salah/mengarang fakta". Cell ini mendemonstrasikan persis situasi tersebut: model ditanya soal aturan yang tidak pernah ada di data training-nya.

## CELL 15 - Ask WITH the handbook  (CORE)

Now... Same question, but the model gets the handbook.

![RAG](https://raw.githubusercontent.com/modafarshouha/mini-rag-workshop/main/assets/RAG.png)

In [33]:
reply, hits = ask(question)
print(reply)
print("\n--- the paragraphs it was given ---")
show(hits)

Based on the handbook, a creator must hold a minimum balance historically around 30,000 Robux before requesting an exchange. 

This information is found in Paragraph 6.

--- the paragraphs it was given ---
score +0.67   paragraph 6:  Developer Exchange, or DevEx, lets qualifying creators convert Robux earned in their ...
score +0.57   paragraph 3:  Robux can be purchased in fixed packages, for example around 400 Robux for 4.99 USD, ...
score +0.51   paragraph 4:  Roblox Premium has three common tiers. Premium 450 gives 450 Robux per month for abou...


menampilkan dua diagram berdampingan: satu dengan tanda silang merah (PROMPT langsung ke LLM tanpa verifikasi, berisiko hallucination) dan satu lagi diagram RAG lengkap. PPT juga menyebutkan solusi untuk hallucination adalah "Giving the model the RIGHT information BEFORE it answers". Cell ini adalah versi "dengan solusi" dari perbandingan tersebut.

## CELL 16 - Measure it  (BONUS)

15 questions where we already know the right paragraph. This is how real teams check whether a search system is any good.

In [34]:
# 15 questions where we already know the right paragraph.
CHECK_QUESTIONS = [
    ("How much does the 800 Robux package cost?", 3),
    ("How many Robux does a Premium 2200 subscription give per month?", 4),
    ("What do only Premium members get to do with items?", 5),
    ("What is the minimum Robux needed for a Developer Exchange payout?", 6),
    ("What scripting language is used in Roblox Studio?", 7),
    ("What is the difference between a Script and a LocalScript?", 8),
    ("What happens to chat for accounts under 13?", 9),
    ("What does a Parental PIN protect?", 10),
    ("What is a Recent Average Price (RAP)?", 12),
    ("What is required to trade Limited items?", 13),
    ("What is the difference between a Game Pass and a Developer Product?", 15),
    ("When did Roblox Corporation go public and on which exchange?", 18),
    ("What rig type supports more detailed avatar animation?", 21),
    ("What event celebrates top Roblox creators each year?", 24),
    ("Can players enjoy Roblox without spending money?", 28),
]

found_count = 0
for question_text, expected in CHECK_QUESTIONS:
    found = [number for number, score, text in search(question_text, k=3)]
    ok = expected in found
    found_count += ok
    print("OK  " if ok else "MISS", f"want p{expected}", "got", found, "|", question_text)

print(f"\nTop-3 hit rate: {found_count} out of {len(CHECK_QUESTIONS)}")

OK   want p3 got [3, 4, 14] | How much does the 800 Robux package cost?
OK   want p4 got [4, 5, 3] | How many Robux does a Premium 2200 subscription give per month?
OK   want p5 got [5, 4, 13] | What do only Premium members get to do with items?
OK   want p6 got [6, 3, 4] | What is the minimum Robux needed for a Developer Exchange payout?
OK   want p7 got [7, 8, 23] | What scripting language is used in Roblox Studio?
OK   want p8 got [8, 7, 15] | What is the difference between a Script and a LocalScript?
OK   want p9 got [9, 22, 10] | What happens to chat for accounts under 13?
OK   want p10 got [10, 9, 22] | What does a Parental PIN protect?
OK   want p12 got [12, 3, 4] | What is a Recent Average Price (RAP)?
OK   want p13 got [12, 13, 5] | What is required to trade Limited items?
OK   want p15 got [15, 14, 23] | What is the difference between a Game Pass and a Developer Product?
OK   want p18 got [18, 1, 2] | When did Roblox Corporation go public and on which exchange?
OK   want p21 

secara eksplisit menyebut "Measuring" sebagai salah satu hal yang ditambahkan sistem RAG nyata, dengan penjelasan yang merujuk langsung ke cell ini: "exactly what the check-questions cell did, but with hundreds of questions." Ini termasuk bagian yang paling jelas dasarnya dari PPT — slide bahkan menyebut cell ini secara spesifik.

## CELL 17 - Lost your connection?

Use the menu: **Runtime -> Run all**. Then run this cell to check that everything came back.

In [18]:
# Lost your connection? Use the menu: Runtime -> Run all. It takes about a minute.
# This cell only CHECKS whether everything is still here.
missing = [
    name
    for name in ["PARAGRAPHS", "model", "PARAGRAPH_VECTORS", "search", "answer", "TIMINGS"]
    if name not in globals()
]

if missing:
    print("MISSING:", ", ".join(missing))
    print("\nFix it with the menu: Runtime -> Run all")
else:
    print("ALL GOOD -", len(PARAGRAPHS), "paragraphs ready. Keep going.")

ALL GOOD - 35 paragraphs ready. Keep going.


menjelaskan bahwa inference (proses model menjawab prompt) berjalan secara real-time dan bergantung pada apa yang sudah "dipelajari" model. Cell ini tidak berkaitan langsung dengan konsep tersebut, melainkan murni pengaman teknis dari sisi notebook — mengecek apakah variabel-variabel penting (PARAGRAPHS, model, PARAGRAPH_VECTORS, search, answer, TIMINGS) masih tersedia di memori kalau sesi Colab sempat terputus.